In [1]:
import utilities

utilities.download_istat()

already downloaded: data/raw/istat/metadata_41_983.xml
already downloaded: data/raw/istat/accidents_2015_2017.csv
already downloaded: data/raw/istat/accidents_2018_2020.csv
already downloaded: data/raw/istat/accidents_2021_2023.csv
already downloaded: data/raw/istat/accidents_2024_2026.csv
all chunks combined: (233538, 16)


In [3]:
utilities.download_situas()

already downloaded: data/raw/situas/situas_comuni_2020.csv


<center>

### First look at the datas 

</center>

In [4]:
# libraries

from bs4 import BeautifulSoup
import pandas as pd

In [5]:
# ISTAT – road accidents

istat = pd.read_csv("data/raw/istat_accidents_all.csv")
print(istat.shape)
istat.head(20)

(233538, 16)


,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2015,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2016,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2017,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2018,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,M,2015,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,IT1:41_983(1.0),A,1001,KILLINJ,M,2016,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,IT1:41_983(1.0),A,1001,KILLINJ,M,2017,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,IT1:41_983(1.0),A,1001,KILLINJ,M,2018,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,IT1:41_983(1.0),A,1001,ROADACC,9,2015,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,IT1:41_983(1.0),A,1001,ROADACC,9,2016,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
istat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 233538 entries, 0 to 233537
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          233538 non-null  object 
 1   FREQ              233538 non-null  object 
 2   REF_AREA          233538 non-null  int64  
 3   DATA_TYPE         233538 non-null  object 
 4   RESULT            233538 non-null  object 
 5   TIME_PERIOD       233538 non-null  int64  
 6   OBS_VALUE         233538 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64

In [7]:
istat.describe()

,REF_AREA,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
count,233538.000000,233538.000000,233538.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,45121.574853,2019.393405,16.914292,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,32530.119459,2.831495,180.715035,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1001.000000,2015.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,16160.000000,2017.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,40011.000000,2019.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,72041.000000,2022.000000,7.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,111107.000000,2024.000000,17306.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# missing values in each column
istat.isna().sum() 



DATAFLOW                 0
FREQ                     0
REF_AREA                 0
DATA_TYPE                0
RESULT                   0
TIME_PERIOD              0
OBS_VALUE                0
OBS_STATUS          233538
NOTE_DS             233538
NOTE_REF_AREA       233538
NOTE_DATA_TYPE      233538
NOTE_RESULT         233538
NOTE_TIME_PERIOD    233538
BASE_PER            233538
UNIT_MEAS           233538
UNIT_MULT           233538
dtype: int64

In [9]:
###Comments

#   233,538 rows and 16 columns, but 9 columns (`OBS_STATUS`, `NOTE_*`, `BASE_PER`, `UNIT_*`) are **completely empty** → we will drop them.
#  `DATAFLOW` and `FREQ` have only one value (dataset id and "A" = annual) → not useful.
#  `REF_AREA` (the municipality code) was read as a **number**: `001001` became `1001`, so we lost the leading zeros. We need to read it as text, otherwise we cannot join it with other data.
#  `OBS_VALUE` is the number we are interested in, but we don't know yet *what* it counts: this depends on `DATA_TYPE` and `RESULT`.


In [10]:
# let's look at the categorical columns
for col in ["DATAFLOW", "FREQ", "DATA_TYPE", "RESULT", "TIME_PERIOD"]:
    print(istat[col].value_counts())
    print("-----")

DATAFLOW
IT1:41_983(1.0)    233538
Name: count, dtype: int64
-----
FREQ
A    233538
Name: count, dtype: int64
-----
DATA_TYPE
KILLINJ    155692
ROADACC     77846
Name: count, dtype: int64
-----
RESULT
F    77846
M    77846
9    77846
Name: count, dtype: int64
-----
TIME_PERIOD
2015    24141
2016    24000
2017    23934
2018    23862
2019    23745
2020    23712
2021    23712
2022    23712
2023    23703
2024    19017
Name: count, dtype: int64
-----


In [11]:
# which combinations of DATA_TYPE and RESULT exist?
istat.groupby(["DATA_TYPE", "RESULT"]).size()

DATA_TYPE  RESULT
KILLINJ    F         77846
           M         77846
ROADACC    9         77846
dtype: int64

In [ ]:
# Looking at the metadata to look for the code list that we need using beautifulsoup

f = open("data/raw/istat/metadata_41_983.xml", encoding="utf-8")
soup = BeautifulSoup(f.read(), "xml")
f.close()

# CL_TIPO_DATO22 = list of the DATA_TYPE codes
# CL_ESITO = list of the RESULT codes

for codelist_id in ["CL_TIPO_DATO22", "CL_ESITO"]:
    codelist = soup.find("Codelist", {"id": codelist_id})
    print(codelist_id)
    for code in codelist.find_all("Code"):
        if code["id"] in ["ROADACC", "KILLINJ", "M", "F", "9"]:
            print("  ", code["id"], "-", code.find("Name").text)

CL_TIPO_DATO22
   KILLINJ - killed and injured
   ROADACC - road accidents with injuries
CL_ESITO
   M - killed
   F - injured
   9 - total


### 2.2 SITUAS – population and area

In [13]:
situas = pd.read_csv("data/raw/situas/situas_comuni_2020.csv", sep=";")
print(situas.shape)
situas.head(20)

(7903, 17)


,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,1,1,201,1001,1001,Agliè,NaN,TO,0,0,2644,2011,"13,1462",2020,2545,2020
1,1,1,1,201,1002,1002,Airasca,NaN,TO,0,0,3819,2011,"15,7393",2020,3633,2020
2,1,1,1,201,1003,1003,Ala di Stura,NaN,TO,0,0,462,2011,"46,3315",2020,459,2020
3,1,1,1,201,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1791,2011,"11,7314",2020,1638,2020
4,1,1,1,201,1006,1006,Almese,NaN,TO,0,0,6303,2011,"17,8756",2020,6355,2020
5,1,1,1,201,1007,1007,Alpette,NaN,TO,0,0,277,2011,"5,6261",2020,235,2020
6,1,1,1,201,1008,1008,Alpignano,NaN,TO,0,0,16893,2011,"11,9193",2020,16484,2020
7,1,1,1,201,1009,1009,Andezeno,NaN,TO,0,0,1966,2011,"7,4860",2020,2013,2020
8,1,1,1,201,1010,1010,Andrate,NaN,TO,0,0,512,2011,"9,3085",2020,488,2020
9,1,1,1,201,1011,1011,Angrogna,NaN,TO,0,0,870,2011,"38,8782",2020,838,2020


In [14]:
situas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7903 entries, 0 to 7902
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Codice Ripartizione geografica  7903 non-null   int64 
 1   Codice Regione                  7903 non-null   int64 
 2   Codice Provincia (Storico)      7903 non-null   int64 
 3   Codice Provincia/Uts            7903 non-null   int64 
 4   Codice Comune (alfanumerico)    7903 non-null   int64 
 5   Codice Comune (numerico)        7903 non-null   int64 
 6   Comune                          7902 non-null   object
 7   Comune (dizione straniera)      124 non-null    object
 8   Sigla automobilistica           7811 non-null   object
 9   Capoluogo di Provincia/Uts      7903 non-null   int64 
 10  Capoluogo di Regione            7903 non-null   int64 
 11  Popolazione legale              7903 non-null   int64 
 12  Anno Censimento                 7903 non-null   

In [15]:
situas.isna().sum()

Codice Ripartizione geografica       0
Codice Regione                       0
Codice Provincia (Storico)           0
Codice Provincia/Uts                 0
Codice Comune (alfanumerico)         0
Codice Comune (numerico)             0
Comune                               1
Comune (dizione straniera)        7779
Sigla automobilistica               92
Capoluogo di Provincia/Uts           0
Capoluogo di Regione                 0
Popolazione legale                   0
Anno Censimento                      0
Superficie (Kmq)                     0
Anno (Superficie)                    0
Popolazione residente                0
Anno (Popolazione residente)         0
dtype: int64

In [17]:
# "Comune (dizione straniera)" is almost empty: which municipalities have it?
foreign = situas[situas["Comune (dizione straniera)"].notna()]
print(len(foreign))
print(foreign["Sigla automobilistica"].value_counts())
foreign[["Comune", "Comune (dizione straniera)"]].head()

124
Sigla automobilistica
BZ    116
TS      4
GO      3
TN      1
Name: count, dtype: int64


,Comune,Comune (dizione straniera)
2648,Aldino/Aldein,Aldein
2649,Andriano/Andrian,Andrian
2650,Anterivo/Altrei,Altrei
2651,Appiano sulla strada del vino/Eppan an der Wei...,Eppan an der Weinstraße
2652,Avelengo/Hafling,Hafling


- `Comune (dizione straniera)` is empty for 7,779 municipalities: this is **expected**, the column is filled only for the 124 municipalities with an official name in another language (German in Bolzano, Slovene in Trieste/Gorizia). The bilingual name is already in `Comune`, so we will drop this column.

In [16]:
# Missing values in Comune(dizione straniera) and Sigla automobilistica

print(situas[situas["Comune"].isna()][["Codice Comune (alfanumerico)", "Comune", "Sigla automobilistica"]])
print(situas[situas["Sigla automobilistica"].isna()][["Comune", "Sigla automobilistica"]].head())

     Codice Comune (alfanumerico) Comune Sigla automobilistica
164                          1168    NaN                    TO
        Comune Sigla automobilistica
5066    Acerra                   NaN
5067  Afragola                   NaN
5068   Agerola                   NaN
5069  Anacapri                   NaN
5070    Arzano                   NaN


In [ ]:
# What we notice:

# - 7,903 rows = all the Italian municipalities existing on 31/12/2020.
# - `Superficie (Kmq)` (area) was read as **text**, because decimals use a comma (`13,1462`) and Rome even has a thousands separator (`1.287,3716`).
# - `Codice Comune (alfanumerico)` lost its leading zeros, like `REF_AREA` in ISTAT.
# - The missing values are **not real missing values**: pandas reads the text `"NA"` as NaN, so the province code of **Napoli (NA)** disappears (92 municipalities), and the municipality called **"None"** (Piedmont) loses its name!
# - Population and area are from **2020** (`Anno (Popolazione residente)` = 2020 for all rows), while `Popolazione legale` comes from the **2011 census**. This is the "year specific" warning of the assignment: we will use the 2020 resident population.
# - Population goes from 29 people (Morterone) to 2.77 million (Rome): very different sizes, we will need to keep this in mind for the rates per inhabitant.
